# HTR-JAND: line-level handwriting recognition demo

HTR-JAND, the line-level handwriting recognizer: a compact Teacher–Student (distilled) CTC model with joint
attention. It reads **one line** of handwriting per image. The models were trained on IAM, RIMES, Bentham,
Saint Gall and Washington plus synthetic lines; the examples below are out-of-distribution public-domain letters.

Weights: CC BY-NC 4.0 (non-commercial, inherited from the training data). Code: MIT.

## 1. Install

In [ ]:
import importlib.util, subprocess, sys

# installs the package (with the optional T5 dependency) unless it is already available
if importlib.util.find_spec("htrjand") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "htrjand[t5] @ git+https://github.com/DocumentRecognitionModels/htrjand.git"])
import htrjand
print("htrjand", htrjand.__version__, htrjand.RELEASE)

## 2. Load a pretrained model from the Hugging Face Hub

`MODEL_ID` is the only setting: a Hub repo id (default) or a local directory holding the repo
(`<variant>/config.json`). Variants: `general` (default, five corpora), `iam`, `rimes`, `bentham`,
`saintgall`, `washington`.

In [ ]:
import os
from htrjand import HTRJANDModel, release

MODEL_ID = os.environ.get("HTRJAND_MODEL_ID", release.DEFAULT_MODEL)
VARIANT = "general"

model = HTRJANDModel.from_pretrained(MODEL_ID, variant=VARIANT, device="cpu")
print(model.config["description"])
print(f"{model.deployed_parameters():,} parameters at inference, input {model.input_size}")

## 3. Load an example image

In [ ]:
import urllib.request
from PIL import Image
from IPython.display import display

EXAMPLES_URL = f"https://raw.githubusercontent.com/{release.GITHUB_REPO}/main/examples/"
EXAMPLES_DIR = os.environ.get("HTRJAND_EXAMPLES_DIR")      # optional local copy of examples/

def example(name):
    if EXAMPLES_DIR:
        return os.path.join(EXAMPLES_DIR, name)
    if not os.path.exists(name):
        urllib.request.urlretrieve(EXAMPLES_URL + name, name)
    return name

path = example("ood_2_gettysburg_a.png")   # Lincoln, Gettysburg Address (Bliss copy, 1864), public domain
display(Image.open(path))

## 4. Preprocess (what the model sees)

Grayscale, polarity check, crop to the ink, then exactly the training geometry: one scale factor into
128 x 1632 (aspect ratio kept), centre padding, inversion (ink bright on dark) and per-image normalisation.
The preview is shown dark-on-light.

In [ ]:
prep = model.preprocess(path)
print("crop box (top, bottom, left, right):", prep.crop_box, "| looks like one line:", prep.is_line, prep.warnings)
display(Image.fromarray(255 - prep.canvas))

## 5. Run HTR (raw CTC)

In [ ]:
r = model.transcribe(path)            # decoder="greedy"
display(Image.open(path))
print("raw CTC   :", r.raw)
print("confidence:", round(r.confidence, 3), "(mean max-softmax over the non-blank frames)")

## 6. Optional decoding: character LM and T5

`decoder="lm"` adds a CTC beam search with a character 8-gram built from **Wikipedia text only**
(CC BY-SA 4.0); `t5=True` adds the gated T5-small corrector (about 240 MB download), which changes a line
only when its confidence passes the threshold chosen on validation. Both can also introduce errors.

In [ ]:
r_lm = model.transcribe(path, decoder="lm", corpus="iam")   # corpus selects the LM language: iam/bentham/washington=en, rimes=fr, saintgall=la
print("raw CTC:", r_lm.raw)
print("char LM:", r_lm.lm)

RUN_T5 = True
if RUN_T5:
    try:
        r_t5 = model.transcribe(path, decoder="lm", corpus="iam", t5=True)
        print("T5     :", r_t5.t5, "| applied:", r_t5.t5_applied, "| T5 confidence:", round(r_t5.t5_confidence, 3))
    except ImportError as e:
        print("T5 needs `pip install transformers`:", e)

## 7. Try your own handwriting

In Colab, upload PNG / JPG / JPEG images of **single text lines** (crop one line; light or dark background
both work). Outside Colab, set `MY_IMAGES` to a list of paths.

In [ ]:
MY_IMAGES = []
try:
    from google.colab import files       # only available in Colab
    MY_IMAGES = list(files.upload())
except ImportError:
    print("Not running in Colab: put your own file paths in MY_IMAGES.")

from htrjand import NotALineError
for name in MY_IMAGES:
    if not name.lower().endswith((".png", ".jpg", ".jpeg")):
        print(name, "skipped: use PNG / JPG / JPEG"); continue
    try:
        res = model.transcribe(name, decoder="lm")
    except NotALineError as e:
        print(name, "->", e); continue
    display(Image.open(name))
    print("raw:", res.raw, "| LM:", res.lm, "| confidence:", round(res.confidence, 3), res.warnings)

## 8. Multiple examples

In [ ]:
names = ["ood_2_gettysburg_b.png", "ood_3_darwin_a.png", "ood_4_balzac_b.png",
         "ood_7_modern_de_a.png", "ood_8_modern_en_b.png"]
corpus = {"ood_4_balzac_b.png": "rimes"}            # French letter: French LM
for n in names:
    p = example(n)
    res = model.transcribe(p, decoder="lm", corpus=corpus.get(n, "iam"))
    display(Image.open(p))
    print(f"{n}\n  raw: {res.raw}\n  LM : {res.lm}\n  confidence {res.confidence:.3f}")

## 9. Limitations

- **Line level only.** Pages and multi-line crops are detected and refused (`force=True` overrides;
  `segment_lines=True` uses a naive projection-profile splitter that is not part of HTR-JAND).
- **Domain shift.** Five benchmark hands/collections + synthetic fonts; other writers, scripts, languages, pens and
  scans (low resolution, stains, show-through) raise the error rate. The example letters show this; their
  transcriptions are the authors' readings, not ground truth, and no error rate is reported on them.
- **Charset and languages.** 108 characters (English, French, Latin corpora); the LM and T5 assume these languages.
- **LM / T5 can hallucinate** plausible but wrong words (names, numbers, historical spellings).
- **Confidence** is not calibrated.
- **Not for** forensic, legal-identity, medical or other high-stakes use; weights are non-commercial (CC BY-NC 4.0).